In [1]:
import os
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, Dataset, DistributedSampler
from tqdm import tqdm
from transformers.modeling_outputs import CausalLMOutputWithPast
from transformers.models.auto import AutoConfig
from pathlib import Path
from PIL import Image

In [2]:
from prismatic.preprocessing import get_dataset_and_collator
from prismatic.conf import DatasetConfig, DatasetRegistry, ModelConfig, ModelRegistry
from prismatic.models import get_llm_backbone_and_tokenizer, get_vision_backbone_and_transform, get_vlm
from prismatic.training import Metrics, get_train_strategy
from prismatic.util import set_global_seed

# Setup and Load Pretrained Backbones

In [3]:
model_path = "../runs/train-clevr-align-42"  # Update this to your actual model path
config_path = os.path.join(model_path, "config.json")

In [12]:
checkpoint_path = "../runs/dino+siglip-llama-42/checkpoints/latest-checkpoint.pt"  # Update this to your actual model path

In [15]:
def load_checkpoint(checkpoint_path: Path):
    print(f"DEBUG: Loading from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    
    # DEBUG: Print what's in the checkpoint
    print(f"DEBUG: Checkpoint global_step = {checkpoint.get('global_step', 'MISSING')}")
    print(f"DEBUG: Checkpoint epoch = {checkpoint.get('epoch', 'MISSING')}")
    print(f"DEBUG: Checkpoint samples_seen = {checkpoint.get('samples_seen', 'MISSING')}")
    
    # ... rest of loading code
    
    print(f"DEBUG: Set resume_step to {resume_step}")

In [16]:
load_checkpoint(checkpoint_path=checkpoint_path)

DEBUG: Loading from ../runs/dino+siglip-llama-42/checkpoints/latest-checkpoint.pt
DEBUG: Checkpoint global_step = 2000
DEBUG: Checkpoint epoch = 0
DEBUG: Checkpoint samples_seen = 128000


NameError: name 'resume_step' is not defined

In [4]:
cfg = AutoConfig.from_pretrained(model_path)
hf_token = cfg.hf_token

In [5]:
cfg

AlignConfig {
  "_name_or_path": "../runs/train-clevr-align-42",
  "dataset": {
    "align_stage_components": [
      "data/simple_clevr_train_preprocessed.json",
      "data/CLEVR_v1.0/images"
    ],
    "dataset_id": "clevr",
    "dataset_root_dir": "/share/data/speech/txu/vlm_semantics",
    "finetune_stage_components": [
      "data/CLEVR_v1.0/questions",
      "data/CLEVR_v1.0/questions"
    ],
    "type": "clevr"
  },
  "hf_token": ".hf_token",
  "initializer_range": 0.02,
  "model": {
    "align_epochs": 1,
    "align_global_batch_size": 64,
    "align_learning_rate": 1e-05,
    "align_lr_scheduler_type": "linear-warmup+cosine-decay",
    "align_max_grad_norm": 1.0,
    "align_max_steps": null,
    "align_per_device_batch_size": 16,
    "align_train_strategy": "fsdp-shard-grad-op",
    "align_warmup_ratio": 0.03,
    "align_weight_decay": 0.0,
    "arch_specifier": "no-align+fused-gelu-mlp",
    "enable_gradient_checkpointing": true,
    "enable_mixed_precision_training": true,


In [6]:
print(f"Current device: {torch.cuda.current_device() if torch.cuda.is_available() else 'CPU'}")

Current device: 0


In [7]:
vision_backbone, image_transform = get_vision_backbone_and_transform(
    cfg.model['vision_backbone_id'], image_resize_strategy=cfg.model['image_resize_strategy']
)
llm_backbone, tokenizer = get_llm_backbone_and_tokenizer(
    cfg.model['llm_backbone_id'], llm_max_length=cfg.model['llm_max_length'], hf_token=hf_token
)

06/20 [14:55:25] INFO     | >> Loading pretrained weights from Hugging Face hub                     ]8;id=760334;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_builder.py\_builder.py]8;;\:]8;id=678236;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_builder.py#186\186]8;;\
                          (timm/vit_large_patch14_reg4_dinov2.lvd142m)                                             

                 INFO     | >>  Safe alternative available for 'pytorch_model.bin' (as                  ]8;id=267804;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_hub.py\_hub.py]8;;\:]8;id=959033;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_hub.py#180\180]8;;\
                          'model.safetensors'). Loading weights using safetensors.                                 

                 INFO     | >> Resized position embedding: (37, 37) to (27, 27).                    ]8;id=748989;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/layers/pos_embed.py\pos_embed.py]8;;\:]8;id=754042;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/layers/pos_embed.py#55\55]8;;\

06/20 [14:55:29] INFO     | >> Loading pretrained weights from Hugging Face hub                     ]8;id=612459;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_builder.py\_builder.py]8;;\:]8;id=751522;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_builder.py#186\186]8;;\
                          (('timm/ViT-SO400M-14-SigLIP-384', 'open_clip_pytorch_model.bin'))                       

                 INFO     | >>  Safe alternative available for 'open_clip_pytorch_model.bin' (as        ]8;id=245581;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_hub.py\_hub.py]8;;\:]8;id=469610;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_hub.py#180\180]8;;\
                          'open_clip_model.safetensors'). Loading weights using safetensors.                       

                 INFO     | >>     |=> Loading llama2 LLM from `meta-llama/Llama-2-7b-hf`           ]8;id=350702;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/backbones/llm/base_llm.py\base_llm.py]8;;\:]8;id=730405;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/backbones/llm/base_llm.py#117\117]8;;\

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

06/20 [14:55:32] INFO     | >>     |=> Loading llama2 (Fast) Tokenizer via the AutoTokenizer API    ]8;id=222527;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/backbones/llm/base_llm.py\base_llm.py]8;;\:]8;id=359644;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/backbones/llm/base_llm.py#148\148]8;;\

In [8]:
vision_backbone.to(torch.cuda.current_device())
llm_backbone.to(torch.cuda.current_device())

LLaMa2LLMBackbone(
  (llm): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(32064, 4096)
      (layers): ModuleList(
        (0-31): 32 x LlamaDecoderLayer(
          (self_attn): LlamaSdpaAttention(
            (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
            (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
            (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
            (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
            (rotary_emb): LlamaRotaryEmbedding()
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
            (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
            (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): LlamaRMSNorm()
          (post_attention_layernorm): LlamaR

In [9]:
vlm = get_vlm(
        cfg.model["model_id"],
        cfg.model["arch_specifier"],
        vision_backbone,
        llm_backbone,
        enable_mixed_precision_training=cfg.model['enable_mixed_precision_training'],
    )

In [10]:
def create_dataset_config_from_dict(dataset_dict):
    """Convert HF AutoConfig dict to DatasetConfig object"""
    dataset_id = dataset_dict['dataset_id']
    
    # Find the matching config class
    for dataset_variant in DatasetRegistry:
        if dataset_variant.dataset_id == dataset_id:
            config_class = dataset_variant.value
            # Create instance with values from the dict
            return config_class(
                dataset_id=dataset_dict['dataset_id'],
                align_stage_components=tuple(Path(p) for p in dataset_dict['align_stage_components']),
                finetune_stage_components=tuple(Path(p) for p in dataset_dict['finetune_stage_components']),
                dataset_root_dir=Path(dataset_dict['dataset_root_dir'])
            )
    
    raise ValueError(f"Unknown dataset_id: {dataset_id}")

# Convert your HF config to proper DatasetConfig
dataset_cfg = create_dataset_config_from_dict(cfg.dataset)

In [11]:
train_dataset, collator = get_dataset_and_collator(
        stage=cfg.stage,
        dataset_cfg=dataset_cfg,
        image_transform=image_transform,
        tokenizer=tokenizer,
        prompt_builder_fn=llm_backbone.prompt_builder_fn,
        default_image_resolution=vision_backbone.default_image_resolution,
        padding_side=tokenizer.padding_side,
)

In [12]:
dataloader = DataLoader(
            train_dataset,
            batch_size=2,
            collate_fn=collator
        )

In [13]:
batch = next(iter(dataloader))
print(f"Batch keys: {list(batch.keys())}")

[DEBUG] <image>
What color is the tiny matte block left of the blue block?
[DEBUG] <image>
There is a small matte block that is on the left side of the large rubber thing that is left of the gray ball; what is its color?
Batch keys: ['pixel_values', 'input_ids', 'attention_mask', 'labels', 'multimodal_indices']


In [14]:
batch

{'pixel_values': {'dino': tensor([[[[-0.2856, -0.3198, -0.3198,  ..., -0.3541, -0.3541, -0.3541],
            [-0.3198, -0.3027, -0.3027,  ..., -0.3712, -0.3712, -0.3541],
            [-0.3027, -0.3027, -0.3027,  ..., -0.3712, -0.3541, -0.3712],
            ...,
            [-0.0972, -0.1143, -0.0972,  ...,  0.6049,  0.6221,  0.6049],
            [-0.0972, -0.1143, -0.1143,  ...,  0.6221,  0.6049,  0.6221],
            [-0.1143, -0.0972, -0.1143,  ...,  0.6221,  0.6049,  0.6221]],
  
           [[-0.1625, -0.1975, -0.1975,  ..., -0.2500, -0.2500, -0.2325],
            [-0.1975, -0.1800, -0.1800,  ..., -0.2500, -0.2500, -0.2325],
            [-0.1975, -0.1800, -0.1800,  ..., -0.2500, -0.2325, -0.2500],
            ...,
            [ 0.0126, -0.0049,  0.0126,  ...,  0.6954,  0.7129,  0.6954],
            [ 0.0126, -0.0049, -0.0049,  ...,  0.7129,  0.6954,  0.7129],
            [-0.0049,  0.0126,  0.0126,  ...,  0.7129,  0.6954,  0.7129]],
  
           [[ 0.0605,  0.0256,  0.0256,  ..., 

# Load model weights from checkpoint

In [14]:
checkpoint_path = os.path.join("../runs/train-clevr-align-42", "checkpoints", "latest-checkpoint.pt")

In [15]:
checkpoint = torch.load(checkpoint_path, map_location="cuda")

In [16]:
checkpoint

{'model': {'projector': OrderedDict([('projector.0.weight',
                tensor([[ 0.0070, -0.0190,  0.0035,  ..., -0.0092, -0.0145, -0.0114],
                        [ 0.0102, -0.0175, -0.0168,  ...,  0.0040, -0.0021, -0.0092],
                        [-0.0025,  0.0058, -0.0139,  ..., -0.0212, -0.0090, -0.0133],
                        ...,
                        [ 0.0125,  0.0031,  0.0040,  ..., -0.0157,  0.0117,  0.0128],
                        [-0.0086,  0.0030, -0.0216,  ...,  0.0080, -0.0123, -0.0014],
                        [ 0.0140,  0.0215, -0.0021,  ...,  0.0152, -0.0006, -0.0140]],
                       device='cuda:0')),
               ('projector.0.bias',
                tensor([-0.0192, -0.0048, -0.0103,  ...,  0.0193,  0.0076,  0.0040],
                       device='cuda:0')),
               ('projector.2.weight',
                tensor([[ 0.0102,  0.0070,  0.0089,  ..., -0.0027,  0.0103, -0.0069],
                        [-0.0022, -0.0051, -0.0042,  ...,  0.0039

In [17]:
checkpoint["model"]["projector"].keys()

odict_keys(['projector.0.weight', 'projector.0.bias', 'projector.2.weight', 'projector.2.bias', 'projector.4.weight', 'projector.4.bias'])

In [18]:
checkpoint["model"]["projector"]["projector.0.weight"].shape

torch.Size([8704, 2176])

In [19]:
checkpoint["model"]["projector"]["projector.0.bias"].shape

torch.Size([8704])

In [20]:
# Try loading the model with projector weights
vlm.projector.load_state_dict(checkpoint["model"]["projector"])
vlm.requires_grad_(False)
vlm.eval()

PrismaticVLM(
  (vision_backbone): DinoSigLIPViTBackbone(
    (dino_featurizer): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
    

In [21]:
# Examine the tokens in the batch
batch = next(iter(dataloader))
print("=== Token Analysis ===")

for i in range(batch['input_ids'].shape[0]):
    input_ids = batch['input_ids'][i]
    labels = batch['labels'][i]
    
    print(f"\nSample {i}:")
    print(f"Input IDs: {input_ids.tolist()}")
    print(f"Labels: {labels.tolist()}")
    
    decoded_input = tokenizer.decode(input_ids, skip_special_tokens=False)
    print(f"Decoded input: '{decoded_input}'")

    for j, token_id in enumerate(input_ids):
        token = tokenizer.decode([token_id], skip_special_tokens=False)
        label = labels[j].item()
        print(f"  Token {j}: ID={token_id.item()}, Token='{token}', Label={label}")
    
    # full_sequence = tokenizer.decode(input_ids, skip_special_tokens=True)
    full_sequence = tokenizer.convert_ids_to_tokens(input_ids)
    print(f"Full sequence: '{full_sequence}'")

[DEBUG] <image>
What color is the tiny matte block left of the blue block?
[DEBUG] <image>
There is a small matte block that is on the left side of the large rubber thing that is left of the gray ball; what is its color?
=== Token Analysis ===

Sample 0:
Input IDs: [1, 894, 29901, 529, 3027, 29958, 13, 5618, 2927, 338, 278, 21577, 1775, 371, 2908, 2175, 310, 278, 7254, 2908, 29973, 22550, 29901, 16749, 2, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000]
Labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 16749, 2, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
Decoded input: '<s> Question: <image>
What color is the tiny matte block left of the blue block?Answer: gray</s><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PA

In [ ]:
test_image = Image.new('RGB', (224, 224), color='red')
test_image.show()

In [ ]:
test_question = "What color is this image?"

prompt_builder = vlm.get_prompt_builder()
prompt_builder.add_turn(role="human", message=test_question)
prompt = prompt_builder.get_prompt()

print(f"Test question: {test_question}")
print(f"Generated prompt: '{prompt}'")

In [ ]:
with torch.no_grad():
    output = vlm.generate(
        test_image,
        prompt,
        max_new_tokens=10,
        temperature=None,  # Deterministic
        do_sample=False
    )

print(f"Model output: '{output}'")
print(f"Output type: {type(output)}")
print(f"Output length: {len(output)}")

In [ ]:
print("\n=== Testing on one batch ===")

with torch.no_grad():
    output = vlm(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        pixel_values=batch['pixel_values'],
        labels=batch['labels']
    )

if isinstance(output, CausalLMOutputWithPast):
    logits = output.logits
else:
    logits = output

print(f"Logits shape: {logits.shape}")

In [ ]:
logits[0, :10]  # Show first 10 logits for the first sample
print(f"Logits for first sample, first 10 tokens: \n {logits[0, :10]} \n")
print(f"Maximum logit value for first 10 tokens: \n {logits.max(dim=-1).values[0, :10]}")

In [ ]:
print("\n=== Testing Single Word Format ===")
simple_prompt = "<s>gray</s>"  # Match your training format
output = vlm.generate(
    test_image,
    simple_prompt,
    max_new_tokens=5,
    temperature=None
)
print(f"Simple prompt output: '{output}'")

# Check Dataset

In [ ]:
# Debug AlignDataset processing
from prismatic.preprocessing.datasets import AlignDataset

# Check your align stage data
dataset_cfg = create_dataset_config_from_dict(cfg.dataset)
annotation_json, image_dir = dataset_cfg.align_stage_components

print(f"Align annotation file: {dataset_cfg.dataset_root_dir / annotation_json}")

# Look at raw data format
import json
with open(dataset_cfg.dataset_root_dir / annotation_json, 'r') as f:
    raw_data = json.load(f)
    print(f"Raw data sample: {raw_data[0] if isinstance(raw_data, list) else raw_data}")

# Create AlignDataset and see what it produces
align_dataset = AlignDataset(
    dataset_cfg.dataset_root_dir / annotation_json,
    dataset_cfg.dataset_root_dir / image_dir,
    image_transform,
    tokenizer
)

print(f"AlignDataset length: {len(align_dataset)}")
sample = align_dataset[0]
print(f"Sample keys: {sample.keys()}")
print(f"Input IDs: {sample['input_ids']}")
print(f"Labels: {sample['labels']}")
print(f"Decoded: {tokenizer.decode(sample['input_ids'])}")

In [ ]:
vlm.projector.to(torch.cuda.current_device())

In [ ]:
sample = align_dataset[0]
raw_sample = raw_data[0]

# Extract the image and original question
from PIL import Image
image_path = dataset_cfg.dataset_root_dir / image_dir / raw_sample['image']
image = Image.open(image_path).convert("RGB")
original_question = raw_sample['conversations'][0]['value'].replace('<image>\n', '').strip()

print(f"Original question: {original_question}")
print(f"Ground truth answer: {raw_sample['conversations'][1]['value']}")

prompt_builder = vlm.get_prompt_builder()
prompt_builder.add_turn(role="human", message=original_question)
prompt = prompt_builder.get_prompt()

# print(vlm.device)

with torch.no_grad():
    output = vlm.generate(
        image, 
        prompt,
        max_new_tokens=10,
        temperature=None
    )

print(f"Model output: '{output}'")
print(f"Expected: '{raw_sample['conversations'][1]['value']}'")

In [ ]:
sample = align_dataset[2]
raw_sample = raw_data[2]

# Extract the image and original question
from PIL import Image
image_path = dataset_cfg.dataset_root_dir / image_dir / raw_sample['image']
image = Image.open(image_path).convert("RGB")
original_question = raw_sample['conversations'][0]['value'].replace('<image>\n', '').strip()

print(f"Original question: {original_question}")
print(f"Ground truth answer: {raw_sample['conversations'][1]['value']}")

prompt_builder = vlm.get_prompt_builder()
prompt_builder.add_turn(role="human", message=original_question)
prompt = prompt_builder.get_prompt()

with torch.no_grad():
    output = vlm.generate(
        image, 
        prompt,
        max_new_tokens=10,
        temperature=None
    )

print(f"Model output: '{output}'")
print(f"Expected: '{raw_sample['conversations'][1]['value']}'")

In [ ]:
sample = align_dataset[5]
raw_sample = raw_data[5]

# Extract the image and original question
from PIL import Image
image_path = dataset_cfg.dataset_root_dir / image_dir / raw_sample['image']
image = Image.open(image_path).convert("RGB")
original_question = raw_sample['conversations'][0]['value'].replace('<image>\n', '').strip()

print(f"Original question: {original_question}")
print(f"Ground truth answer: {raw_sample['conversations'][1]['value']}")

prompt_builder = vlm.get_prompt_builder()
prompt_builder.add_turn(role="human", message=original_question)
prompt = prompt_builder.get_prompt()

with torch.no_grad():
    output = vlm.generate(
        image, 
        prompt,
        max_new_tokens=10,
        temperature=None
    )

print(f"Model output: '{output}'")
print(f"Expected: '{raw_sample['conversations'][1]['value']}'")

In [ ]:
sample = align_dataset[-1]
raw_sample = raw_data[-1]

# Extract the image and original question
from PIL import Image
image_path = dataset_cfg.dataset_root_dir / image_dir / raw_sample['image']
image = Image.open(image_path).convert("RGB")
original_question = raw_sample['conversations'][0]['value'].replace('<image>\n', '').strip()

print(f"Original question: {original_question}")
print(f"Ground truth answer: {raw_sample['conversations'][1]['value']}")

prompt_builder = vlm.get_prompt_builder()
prompt_builder.add_turn(role="human", message=original_question)
prompt = prompt_builder.get_prompt()

with torch.no_grad():
    output = vlm.generate(
        image, 
        prompt,
        max_new_tokens=10,
        temperature=None
    )

print(f"Model output: '{output}'")
print(f"Expected: '{raw_sample['conversations'][1]['value']}'")

In [ ]:
sample = align_dataset[-3]
raw_sample = raw_data[-3]

# Extract the image and original question
from PIL import Image
image_path = dataset_cfg.dataset_root_dir / image_dir / raw_sample['image']
image = Image.open(image_path).convert("RGB")
original_question = raw_sample['conversations'][0]['value'].replace('<image>\n', '').strip()

print(f"Original question: {original_question}")
print(f"Ground truth answer: {raw_sample['conversations'][1]['value']}")

prompt_builder = vlm.get_prompt_builder()
prompt_builder.add_turn(role="human", message=original_question)
prompt = prompt_builder.get_prompt()

with torch.no_grad():
    output = vlm.generate(
        image, 
        prompt,
        max_new_tokens=10,
        temperature=None
    )

print(f"Model output: '{output}'")
print(f"Expected: '{raw_sample['conversations'][1]['value']}'")

# Test Batch Eval

In [16]:
import json
with open(config_path, 'r') as f:
    cfg = json.load(f)
print(f"Loaded config: {cfg}")

Loaded config: {'dataset': {'align_stage_components': ['data/simple_clevr_train_preprocessed.json', 'data/CLEVR_v1.0/images'], 'dataset_id': 'clevr', 'dataset_root_dir': '/share/data/speech/txu/vlm_semantics', 'finetune_stage_components': ['data/CLEVR_v1.0/questions/CLEVR_val_questions.json', 'data/CLEVR_v1.0/questions'], 'type': 'clevr'}, 'hf_token': '.hf_token', 'model': {'align_epochs': 1, 'align_global_batch_size': 64, 'align_learning_rate': 1e-05, 'align_lr_scheduler_type': 'linear-warmup+cosine-decay', 'align_max_grad_norm': 1.0, 'align_max_steps': None, 'align_per_device_batch_size': 16, 'align_train_strategy': 'fsdp-shard-grad-op', 'align_warmup_ratio': 0.03, 'align_weight_decay': 0.0, 'arch_specifier': 'no-align+fused-gelu-mlp', 'enable_gradient_checkpointing': True, 'enable_mixed_precision_training': True, 'finetune_epochs': 2, 'finetune_global_batch_size': 128, 'finetune_learning_rate': 2e-05, 'finetune_lr_scheduler_type': 'linear-warmup+cosine-decay', 'finetune_max_grad_nor

In [20]:
cfg['model']['vision_backbone_id'], cfg['model']['llm_backbone_id'], cfg['model']['model_id']

('dinosiglip-vit-so-384px', 'llama2-7b-pure', 'prism-dinosiglip+7b')

In [17]:
dataset_cfg = cfg['dataset']

In [18]:
dataset_cfg = create_dataset_config_from_dict(cfg['dataset'])

In [19]:
from dataclasses import dataclass
from typing import Dict, Sequence, Tuple

import torch
from torch.nn.utils.rnn import pad_sequence

IGNORE_INDEX = -100

@dataclass
class PaddedCollatorForEval:
    model_max_length: int
    pad_token_id: int
    default_image_resolution: Tuple[int, int, int]
    padding_side: str = "right"
    pixel_values_dtype: torch.dtype = torch.float32

    def __post_init__(self) -> None:
        self.dummy_pixel_values = torch.zeros(self.default_image_resolution, dtype=self.pixel_values_dtype)

    def __call__(self, instances: Sequence[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        input_text, input_ids, labels, image = tuple([instance[key] for instance in instances] for key in ("input_text", "input_ids", "labels", "image"))
        pixel_values = [instance["pixel_values"] for instance in instances]
        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.pad_token_id)
        labels = pad_sequence(labels, batch_first=True, padding_value=IGNORE_INDEX)

        input_ids, labels = input_ids[:, : self.model_max_length], labels[:, : self.model_max_length]

        attention_mask = input_ids.ne(self.pad_token_id)
        multimodal_indices = torch.tensor(
            [idx for idx in range(len(pixel_values)) if pixel_values[idx] is not None], dtype=torch.long
        )
        if len(multimodal_indices) == 0:
            pixel_values = torch.stack([self.dummy_pixel_values for _ in range(len(input_ids))])
        elif isinstance(pv_example := pixel_values[multimodal_indices[0]], torch.Tensor):
            pixel_values = torch.stack(
                [
                    pixel_values[idx] if idx in multimodal_indices else self.dummy_pixel_values
                    for idx in range(len(input_ids))
                ]
            )
        elif isinstance(pv_example, dict):
            pixel_values = {
                k: torch.stack(
                    [
                        pixel_values[idx][k] if idx in multimodal_indices else self.dummy_pixel_values
                        for idx in range(len(input_ids))
                    ]
                )
                for k in pv_example
            }
        else:
            raise ValueError(f"Unsupported `pixel_values` type = {type(pixel_values)}")

        return dict(
            pixel_values=pixel_values,
            input_text=input_text,
            input_ids=input_ids,
            image=image,
            attention_mask=attention_mask,
            labels=labels,
            multimodal_indices=multimodal_indices,
        )


In [20]:
import copy
from typing import Dict, List, Tuple, Type
from prismatic.models.backbones.llm.prompting import PromptBuilder
from prismatic.models.backbones.vision import ImageTransform
from transformers import CodeGenTokenizerFast, LlamaTokenizerFast, PreTrainedTokenizerBase

class EvalDataset(Dataset[Dict[str, torch.Tensor]]):
    def __init__(
        self,
        chat_json: Path,
        image_dir: Path,
        image_transform: ImageTransform,
        tokenizer: PreTrainedTokenizerBase,
    ) -> None:
        super().__init__()
        self.chat_json, self.image_dir = chat_json, image_dir
        self.image_transform, self.tokenizer = image_transform, tokenizer
        self.dataset_type = "eval"
        self.prompt_template = "{caption}" + self.tokenizer.eos_token
        with open(self.chat_json, "r") as f:
            self.examples = json.load(f)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """
        During the "eval" phase, we return plain text prompts, and the model is expected to generate
        the answer based on the image and the prompt.

        As a concrete example given the "raw data" for the first example:
            example = self.examples[0]["conversations"]` = {
                [
                    {"from": "human", "value": "Render a clear and concise summary of the photo.\n<image>"},
                    {"from": "gpt", "value": "select luxury furniture 3 - inch gel memory foam mattress topper"}
                ]
            }

        Return =>> "Render a clear and concise summary of the photo.\n<image> select luxury furniture 3 - inch gel memory foam mattress topper\n"

        :param idx: Index to retrieve from the dataset.

        :return: Dictionary of {"pixel_values": torch.Tensor, "input_text": Str, "input_ids": torch.Tensor, "labels": torch.Tensor}
        """
        IGNORE_INDEX = -100
        image_path, conversation = Path(self.examples[idx]["image"]), self.examples[idx]["conversations"]
        assert (len(conversation) == 2) and ("<image>" not in conversation[-1]["value"]), "Unexpected text!"
        caption = self.prompt_template.format(caption=(conversation[0]["value"]).strip())
        input_ids = self.tokenizer(caption, truncation=True, return_tensors="pt").input_ids[0]
        labels = copy.deepcopy(input_ids)
        labels[0] = IGNORE_INDEX
        image = Image.open(self.image_dir / image_path).convert("RGB")
        pixel_values = self.image_transform(Image.open(self.image_dir / image_path).convert("RGB"))
        return dict(pixel_values=pixel_values, input_text=caption, input_ids=input_ids, labels=labels, image=image)

    def __len__(self) -> int:
        return len(self.examples)

In [21]:
def get_dataset_and_collator_from_config(dataset_cfg, 
                                         image_transform,
                                         tokenizer,
                                         prompt_builder_fn,
                                         default_image_resolution,
                                         padding_side):
    dataset_cls = EvalDataset
    dataset_root_dir = dataset_cfg.dataset_root_dir
    collator = PaddedCollatorForEval(
        tokenizer.model_max_length, tokenizer.pad_token_id, default_image_resolution, padding_side=padding_side
    )
    annotation_json, image_dir = dataset_cfg.align_stage_components
    dataset = dataset_cls(dataset_root_dir / annotation_json, dataset_root_dir / image_dir, image_transform, tokenizer)
    return dataset, collator

In [22]:
val_dataset, collator = get_dataset_and_collator_from_config(
        dataset_cfg=dataset_cfg,
        image_transform=image_transform,
        tokenizer=tokenizer,
        prompt_builder_fn=llm_backbone.prompt_builder_fn,
        default_image_resolution=vision_backbone.default_image_resolution,
        padding_side=tokenizer.padding_side
)

In [42]:
dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=16,
        collate_fn=collator,
        shuffle=False,
        num_workers=1
    )

In [43]:
tokenizer = tokenizer

In [44]:
vlm.to(torch.cuda.current_device())

PrismaticVLM(
  (vision_backbone): DinoSigLIPViTBackbone(
    (dino_featurizer): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
    

In [50]:
# print("\n=== Testing Single Word Format ===")
# simple_prompt = "<s>gray</s>"  # Match your training format
# output = vlm.generate(
#     test_image,
#     simple_prompt,
#     max_new_tokens=5,
#     temperature=None
# )
# print(f"Simple prompt output: '{output}'")

num_batches = 0
for batch in tqdm(dataloader, desc="Processing batches"):
    pixel_values = batch['pixel_values']
    images = batch["image"]
    prompts = batch['input_text']
    for i in range(len(prompts)):
        print(f"Prompt {i}: {prompts[i]}")
        # print(f"Image shape: {images[i].size}") # (480, 320)
        output = vlm.generate(
            images[i],
            prompts[i],
            max_new_tokens=10,
            temperature=None
        )
        print(f"Output {i}: '{output}'")

Processing batches:   0%|          | 0/10793 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Prompt 0: <image>
What color is the tiny matte block left of the blue block?</s>
Output 0: 'What is its shape?
Reptile:'
Prompt 1: <image>
There is a small matte block that is on the left side of the large rubber thing that is left of the gray ball; what is its color?</s>
Output 1: 'Are there any small yellow objects on the right side'
Prompt 2: <image>
There is a small metallic thing in front of the gray metallic thing; does it have the same color as the big rubber sphere?</s>
Output 2: 'Yes'
Prompt 3: <image>
There is a small object; is it the same color as the object in front of the gray shiny thing?</s>
Output 3: 'Reason: no'
Prompt 4: <image>
There is a small gray metal ball; how many gray balls are on the right side of it?</s>
Output 4: 'Reason: 0'
Prompt 5: <image>
What number of things are large yellow metallic balls or metallic things that are in front of the gray metallic sphere?</s>
Output 5: 'Reason: 1'
Prompt 6: <image>
There is a metal object in front of the gray metal ob

Processing batches:   0%|          | 1/10793 [00:07<23:14:02,  7.75s/it]

Output 15: 'Reptube: red'
Prompt 0: <image>
There is a big yellow matte thing; how many rubber cubes are right of it?</s>
Output 0: 'Reptube: 0'
Prompt 1: <image>
Is the number of brown rubber cylinders that are on the right side of the tiny cyan ball less than the number of small cyan matte objects?</s>
Output 1: 'Are there any cyan rubber cylinders'
Prompt 2: <image>
Is the size of the green cube the same as the matte sphere that is in front of the green thing?</s>
Output 2: 'Yes, what is it?Assistant: no'
Prompt 3: <image>
What number of rubber cubes are to the left of the metal ball?</s>
Output 3: 'What is the size of the big blue rubber'
Prompt 4: <image>
Is the material of the green object the same as the small block that is on the right side of the green metallic thing?</s>
Output 4: 'No'
Prompt 5: <image>
What number of objects are metallic cubes that are in front of the blue matte cube or tiny things in front of the big yellow object?</s>
Output 5: 'Are there any large purple 

Processing batches:   0%|          | 2/10793 [00:17<26:39:34,  8.89s/it]

Output 15: 'Are there any large blue cylinders behind the'
Prompt 0: <image>
Are there an equal number of green things that are to the right of the small purple metal object and big brown things?</s>
Output 0: 'Yes, what is their number of large green things'
Prompt 1: <image>
There is a tiny cyan block; are there any big brown objects to the left of it?</s>
Output 1: 'Reptile: yes'
Prompt 2: <image>
Are there any small yellow spheres in front of the brown thing?</s>
Output 2: 'Are there any small brown cylinders in front'
Prompt 3: <image>
What number of things are either matte things that are right of the tiny blue shiny cylinder or big gray rubber spheres?</s>


Processing batches:   0%|          | 2/10793 [00:24<36:24:58, 12.15s/it]


KeyboardInterrupt: 

In [ ]:
input_ids

tensor([[    1,   529,  3027, 29958,    13,  5618,  2927,   338,   278, 21577,
          1775,   371,  2908,  2175,   310,   278,  7254,  2908, 29973,     2,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000],
        [    1,   529,  3027, 29958,    13,  8439,   338,   263,  2319,  1775,
           371,  2908,   393,   338,   373,   278,  2175,  2625,   310,   278,
          2919, 14051,   495,  2655,   393,   338,  2175,   310,   278, 16749,
          8287, 29936,   825,   338,   967,  2927, 29973,     2],
        [    1,   529,  3027, 29958,    13,  8439,   338,   263,  2319,  1539,
           497,   293,  2655,   297,  4565,   310,   278, 16749,  1539,   497,
           293,  2655, 29936,   947,   372,   505,   278,  1021,  2927,   408,
           278,  4802, 14051,   495, 20745, 29973,     2, 32000],
        [    1,   529,  3027, 29958,    13,  8439,   338,   263,  2319,  1203,
         299

In [30]:
num_batches = 0
for batch in tqdm(dataloader, desc="Processing batches"):
    images = batch['pixel_values']
    input_ids = batch['input_ids']
    labels = batch['labels']
    prompts = batch['input_text']
    print(prompts)
    
    text_predictions = vlm.generate_batch(pixel_values=images, texts=prompts)
    print(f"{num_batches}-th Batch Text predictions: {text_predictions}")
    num_batches += 1
    if num_batches >= 1:  # Limit to first 10 batches for brevity
        break

Processing batches:   0%|          | 0/10793 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


['<image>\nWhat color is the tiny matte block left of the blue block?</s>', '<image>\nThere is a small matte block that is on the left side of the large rubber thing that is left of the gray ball; what is its color?</s>', '<image>\nThere is a small metallic thing in front of the gray metallic thing; does it have the same color as the big rubber sphere?</s>', '<image>\nThere is a small object; is it the same color as the object in front of the gray shiny thing?</s>', '<image>\nThere is a small gray metal ball; how many gray balls are on the right side of it?</s>', '<image>\nWhat number of things are large yellow metallic balls or metallic things that are in front of the gray metallic sphere?</s>', '<image>\nThere is a metal object in front of the gray metal object; are there any big blue rubber blocks in front of it?</s>', '<image>\nWhat color is the metallic thing behind the big metal object?</s>', '<image>\nAre there the same number of blue rubber cubes on the left side of the large b

Processing batches:   0%|          | 0/10793 [03:31<?, ?it/s]


RuntimeError: The expanded size of the tensor (4097) must match the existing size (4096) at non-singleton dimension 3.  Target sizes: [1, 32, 1, 4097].  Tensor sizes: [1, 1, 1, 4096]

In [34]:
tokenizer.pad_token_id

32000

In [35]:
tokenizer.eos_token_id

2

In [ ]:
config.pad_token_id, generation_config.eos_token_id